import os
import shutil
import sys

# --- 1. PURGE THE CACHE (CRITICAL) ---
# This deletes the corrupt "torch-2.10" file so pip can never find it again.
print("Purging pip cache...")
!pip cache purge

# --- 2. SETUP ENVIRONMENT ---
os.environ["PYTHONPATH"] = f"/kaggle/working:{os.environ.get('PYTHONPATH', '')}"
os.environ["MAX_JOBS"] = "4"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5;8.0;8.6"

# --- 3. WIPE EVERYTHING ---
print("Wiping all libraries...")
folders = [
    '/kaggle/working/torch', '/kaggle/working/natten', 
    '/kaggle/working/mamba_ssm', '/kaggle/working/causal_conv1d',
    '/kaggle/working/wheels', '/kaggle/working/torch-2.10.0.dist-info'
]
for folder in folders:
    if os.path.exists(folder):
        shutil.rmtree(folder)
os.makedirs("/kaggle/working/wheels", exist_ok=True)

# --- 4. INSTALL PYTORCH 2.5 (FORCE) ---
print("⬇️  Installing PyTorch 2.5...")
# We use --no-cache-dir to force it to download the REAL 2.5, not the cached ghost
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 \
    --index-url https://download.pytorch.org/whl/cu124 \
    --target=/kaggle/working/ \
    --force-reinstall \
    --no-cache-dir

# --- 5. INSTALL NATTEN (FORCE) ---
print("⬇️  Installing Natten...")
!pip install \
    https://shi-labs.com/natten/wheels/cu124/torch2.5.0/natten-0.17.5%2Btorch250cu124-cp312-cp312-linux_x86_64.whl \
    --index-url https://download.pytorch.org/whl/cu124 \
    --trusted-host shi-labs.com \
    --target=/kaggle/working/

# --- 6. COMPILE MAMBA ---
print("\n⚙️  Compiling Mamba (Takes ~7 mins)...")
!pip wheel causal-conv1d>=1.4.0 mamba-ssm>=2.2.2 \
    --no-build-isolation \
    --no-deps \
    --wheel-dir=/kaggle/working/wheels

# --- 7. INSTALL MAMBA ---
print("\n📦 Installing Mamba Wheels...")
!pip install /kaggle/working/wheels/causal_conv1d*.whl /kaggle/working/wheels/mamba_ssm*.whl \
    --target=/kaggle/working/ \
    --no-deps

print("SUCCESS!")

!pip install \
    transformers==4.46.3 \
    tokenizers==0.20.3\
    timm>=0.9.0 \
    einops>=0.7.0 \
    fvcore>=0.1.5 \
    medmnist \
    tensorboardX \
    torchattacks \
    --target=/kaggle/working/ \
    --no-deps

print("All libraries installed.")

In [ ]:
import sys
import os

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

import torch
import natten
import mamba_ssm

print(f"Mamba Version: {mamba_ssm.__version__}")
print(f"✅ Torch Version:  {torch.__version__}")
print(f"✅ Natten Version: {natten.__version__}")
print(f"📂 Torch Location: {torch.__file__}") 



In [ ]:
cd /kaggle/input/medhybrid

In [ ]:
import os
import sys
import numpy as np
import time

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import PIL
from collections import OrderedDict
from copy import deepcopy

import torchvision.utils
from torchvision import models
import torchvision.datasets as dsets
import torchvision.transforms as transforms
from torchsummary import summary
from tensorboardX import SummaryWriter


from tqdm import tqdm
from tqdm import trange
import medmnist
from medmnist import INFO, Evaluator

import torchattacks
from torchattacks import PGD, FGSM

try:
    from thop import profile as thop_profile
    from thop import clever_format
    THOP_AVAILABLE = True
except ImportError:
    THOP_AVAILABLE = False
    print("thop chưa cài. Chạy: !pip install thop -q")
 
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    cohen_kappa_score,
    matthews_corrcoef,
    classification_report,
)


In [ ]:
from MedCobra import medcobra_t


In [ ]:
def count_parameters(model):
    """
    Đếm số tham số của model.
    Returns dict: total_params, trainable_params, total_M, trainable_M
    """
    total      = sum(p.numel() for p in model.parameters())
    trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        'total_params':     total,
        'trainable_params': trainable,
        'total_M':          total     / 1e6,
        'trainable_M':      trainable / 1e6,
    }
 
 
def compute_flops(model, input_size, device):
    """
    Tính FLOPs và MACs bằng thop.profile().
    input_size: (C, H, W), ví dụ (3, 224, 224)
    Returns dict hoặc None nếu thop chưa cài.
    """
    if not THOP_AVAILABLE:
        return None
    model.eval()
    dummy = torch.randn(1, *input_size).to(device)
    try:
        macs_raw, _ = thop_profile(model, inputs=(dummy,), verbose=False)
        flops_raw   = macs_raw * 2   # FLOPs ≈ 2 × MACs
        macs_str,  _ = clever_format([macs_raw,  1], '%.3f')
        flops_str, _ = clever_format([flops_raw, 1], '%.3f')
        return {
            'macs':      macs_raw,
            'flops':     flops_raw,
            'macs_str':  macs_str,
            'flops_str': flops_str,
        }
    except Exception as e:
        print(f"compute_flops thất bại: {e}")
        return None
 
 
def measure_inference_time(model, input_size, device, n_warmup=50, n_runs=200):
    """
    Đo inference time (ms/image) với batch_size=1.
    Dùng CUDA Events nếu có GPU, ngược lại dùng perf_counter.
    Quy trình: warmup n_warmup lần → đo n_runs lần → trả mean ± std.
    Returns dict: mean_ms, std_ms, min_ms, max_ms
    """
    model.eval()
    dummy    = torch.randn(1, *input_size).to(device)
    use_cuda = (device.type == 'cuda')
 
    # Warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
    if use_cuda:
        torch.cuda.synchronize()
 
    # Đo
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            if use_cuda:
                s = torch.cuda.Event(enable_timing=True)
                e = torch.cuda.Event(enable_timing=True)
                s.record()
                _ = model(dummy)
                e.record()
                torch.cuda.synchronize()
                times.append(s.elapsed_time(e))          # ms
            else:
                t0 = time.perf_counter()
                _ = model(dummy)
                times.append((time.perf_counter() - t0) * 1000)  # ms
 
    times = np.array(times)
    return {
        'mean_ms': float(np.mean(times)),
        'std_ms':  float(np.std(times)),
        'min_ms':  float(np.min(times)),
        'max_ms':  float(np.max(times)),
    }
 
 
def profile_model(model, input_size, device):
    """
    Gọi cả 3 hàm trên, in kết quả ra console và trả về dict tổng hợp.
    Gọi 1 lần sau khi build model, trước training loop.
    """
    print('\n' + '─' * 55)
    print('MODEL PERFORMANCE PROFILE')
    print('─' * 55)
    result = {}
 
    # 1. Parameters
    p = count_parameters(model)
    result.update(p)
    print(f"  Total params     : {p['total_params']:,}  ({p['total_M']:.3f} M)")
    print(f"  Trainable params : {p['trainable_params']:,}  ({p['trainable_M']:.3f} M)")
 
    # 2. FLOPs / MACs
    f = compute_flops(model, input_size, device)
    if f:
        result.update(f)
        print(f"  MACs             : {f['macs_str']}  (raw: {f['macs']:.2e})")
        print(f"  FLOPs (~2×MACs)  : {f['flops_str']}  (raw: {f['flops']:.2e})")
    else:
        print("  MACs / FLOPs     : N/A (thop not installed)")
 
    # 3. Inference time
    print(f"  Đang đo inference time ({200} runs, batch=1)...")
    t = measure_inference_time(model, input_size, device)
    result.update({f'infer_{k}': v for k, v in t.items()})
    print(f"  Inference time   : {t['mean_ms']:.3f} ± {t['std_ms']:.3f} ms/image")
    print(f"                     (min: {t['min_ms']:.3f}  max: {t['max_ms']:.3f} ms)")
 
    print('─' * 55 + '\n')
    return result
 
 
def save_profile_to_txt(profile, output_root, data_flag):
    """Ghi hardware profile vào file log (chạy 1 lần)."""
    with open(os.path.join(output_root, f'{data_flag}_log.txt'), 'a') as f:
        f.write('\n=== MODEL PERFORMANCE PROFILE ===\n')
        f.write(f"  Total params     : {profile.get('total_params', 'N/A'):,}"
                f"  ({profile.get('total_M', 0):.3f} M)\n")
        f.write(f"  Trainable params : {profile.get('trainable_params', 'N/A'):,}"
                f"  ({profile.get('trainable_M', 0):.3f} M)\n")
        if 'macs_str' in profile:
            f.write(f"  MACs             : {profile['macs_str']}\n")
            f.write(f"  FLOPs            : {profile['flops_str']}\n")
        if 'infer_mean_ms' in profile:
            f.write(f"  Inference time   : {profile['infer_mean_ms']:.3f}"
                    f" ± {profile['infer_std_ms']:.3f} ms/image\n")
        f.write('\n')
 
 
 
def compute_extra_metrics(y_true, y_score, task, n_classes):
    """
    Tính các chỉ số phân loại ngoài AUC/ACC.
 
    Parameters
    ----------
    y_true    : np.ndarray (N,)    — nhãn thật (integer)
    y_score   : np.ndarray (N, C)  — xác suất từng lớp (sau softmax/sigmoid)
    task      : str                — 'multi-label, binary-class' | 'binary-class' | 'multi-class'
    n_classes : int
 
    Returns
    -------
    dict: f1_macro, f1_weighted, f1_per_class,
          precision_macro, precision_weighted,
          recall_macro, recall_weighted,
          sensitivity, specificity_macro, specificity_per_class,
          cohen_kappa, mcc, confusion_matrix
    """
    metrics = {}
 
    if task == 'multi-label, binary-class':
        y_pred = (y_score >= 0.5).astype(int)
 
        metrics['f1_macro']           = f1_score(y_true, y_pred, average='macro',    zero_division=0)
        metrics['f1_weighted']        = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics['f1_per_class']       = f1_score(y_true, y_pred, average=None,       zero_division=0).tolist()
        metrics['precision_macro']    = precision_score(y_true, y_pred, average='macro',    zero_division=0)
        metrics['precision_weighted'] = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics['recall_macro']       = recall_score(y_true, y_pred, average='macro',    zero_division=0)
        metrics['recall_weighted']    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics['sensitivity']        = metrics['recall_macro']
        # Các chỉ số dưới không trivial cho multi-label
        metrics['specificity_macro']  = None
        metrics['cohen_kappa']        = None
        metrics['mcc']                = None
        metrics['confusion_matrix']   = None
 
    else:
        y_pred = np.argmax(y_score, axis=1)
 
        metrics['f1_macro']           = f1_score(y_true, y_pred, average='macro',    zero_division=0)
        metrics['f1_weighted']        = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics['f1_per_class']       = f1_score(y_true, y_pred, average=None,       zero_division=0).tolist()
        metrics['precision_macro']    = precision_score(y_true, y_pred, average='macro',    zero_division=0)
        metrics['precision_weighted'] = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics['recall_macro']       = recall_score(y_true, y_pred, average='macro',    zero_division=0)
        metrics['recall_weighted']    = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics['sensitivity']        = metrics['recall_macro']   # Sensitivity = Recall
 
        # Specificity (TNR) per class → macro average
        cm = confusion_matrix(y_true, y_pred)
        metrics['confusion_matrix'] = cm.tolist()
        specificities = []
        for i in range(len(cm)):
            tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
            fp = cm[:, i].sum() - cm[i, i]
            specificities.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
        metrics['specificity_macro']    = float(np.mean(specificities))
        metrics['specificity_per_class'] = specificities
 
        metrics['cohen_kappa'] = cohen_kappa_score(y_true, y_pred)
        metrics['mcc']         = matthews_corrcoef(y_true, y_pred)
 
    return metrics
 
 
def format_extra_metrics(metrics, prefix=''):
    """Trả về string in đẹp để in console hoặc ghi file."""
    p     = f'[{prefix}] ' if prefix else ''
    lines = [
        f"{p}F1 Macro     : {metrics['f1_macro']:.5f}",
        f"{p}F1 Weighted  : {metrics['f1_weighted']:.5f}",
        f"{p}Precision (M): {metrics['precision_macro']:.5f}",
        f"{p}Recall / Sens: {metrics['recall_macro']:.5f}",
    ]
    if metrics.get('specificity_macro') is not None:
        lines.append(f"{p}Specificity  : {metrics['specificity_macro']:.5f}")
    if metrics.get('cohen_kappa') is not None:
        lines.append(f"{p}Cohen Kappa  : {metrics['cohen_kappa']:.5f}")
    if metrics.get('mcc') is not None:
        lines.append(f"{p}MCC          : {metrics['mcc']:.5f}")
    return '\n'.join(lines)
 
 
def save_metrics_to_txt(log_dict_epoch, extra_train, extra_val, extra_test,
                        epoch, data_flag, output_root):
    """Ghi chỉ số phân loại của epoch hiện tại vào file full_log."""
    with open(os.path.join(output_root, f'{data_flag}_full_log.txt'), 'a') as f:
        f.write(f'\n=== Epoch {epoch} ===\n')
        for split, extra in [('TRAIN', extra_train), ('VAL', extra_val), ('TEST', extra_test)]:
            f.write(f'\n-- {split} --\n')
            f.write(format_extra_metrics(extra, prefix=split) + '\n')
            if extra.get('f1_per_class'):
                per = [f'{v:.4f}' for v in extra['f1_per_class']]
                f.write(f'[{split}] F1 per class : {per}\n')
 
 
 
iteration = 0   # global step cho TensorBoard
 
def train(model, train_loader, task, criterion, optimizer, device, writer):
    global iteration
    total_loss = []
    model.train()
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        outputs = model(inputs.to(device))
 
        if task == 'multi-label, binary-class':
            targets = targets.to(torch.float32).to(device)
            loss    = criterion(outputs, targets)
        else:
            targets = torch.squeeze(targets, 1).long().to(device)
            loss    = criterion(outputs, targets)
 
        total_loss.append(loss.item())
        writer.add_scalar('train_loss_logs', loss.item(), iteration)
        iteration += 1
        loss.backward()
        optimizer.step()
 
    return sum(total_loss) / len(total_loss)
 
 
def test(model, evaluator, data_loader, task, criterion, device, run, save_folder=None):
    """
    Returns
    -------
    [test_loss, auc, acc] : list
    y_true_np             : np.ndarray (N,)    — nhãn thật
    y_score_np            : np.ndarray (N, C)  — xác suất dự đoán
    """
    model.eval()
    total_loss  = []
    y_score     = torch.tensor([]).to(device)
    y_true_list = []
 
    with torch.no_grad():
        for inputs, targets in data_loader:
            outputs = model(inputs.to(device))
 
            if task == 'multi-label, binary-class':
                targets = targets.to(torch.float32).to(device)
                loss    = criterion(outputs, targets)
                outputs = nn.Sigmoid()(outputs)
                y_true_list.append(targets.cpu().numpy())
            else:
                targets = torch.squeeze(targets, 1).long().to(device)
                loss    = criterion(outputs, targets)
                outputs = nn.Softmax(dim=1)(outputs)
                y_true_list.append(targets.cpu().numpy())
                targets = targets.float().resize_(len(targets), 1)
 
            total_loss.append(loss.item())
            y_score = torch.cat((y_score, outputs), 0)
 
    y_score_np = y_score.detach().cpu().numpy()
    y_true_np  = np.concatenate(y_true_list, axis=0)
    auc, acc   = evaluator.evaluate(y_score_np, save_folder, run)
    test_loss  = sum(total_loss) / len(total_loss)
 
    return [test_loss, auc, acc], y_true_np, y_score_np


In [ ]:

from IPython.display import FileLink, display

def main(data_flag, output_root, num_epochs, gpu_ids, batch_size, size,
         download, model_flag, resize, as_rgb, model_path, run, resume_path=None):

    lr         = 0.001
    gamma      = 0.1
    milestones = [0.5 * num_epochs, 0.75 * num_epochs]

    info       = INFO[data_flag]
    task       = info['task']
    n_channels = 3 if as_rgb else info['n_channels']
    n_classes  = len(info['label'])
    DataClass  = getattr(medmnist, info['python_class'])

    # ── GPU setup ─────────────────────────────────────────────────────────────
    str_ids = gpu_ids.split(',')
    gpu_ids = [int(i) for i in str_ids if int(i) >= 0]
    if gpu_ids:
        os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_ids[0])
    device = torch.device(f'cuda:{gpu_ids[0]}') if gpu_ids else torch.device('cpu')

    # ── Output folder ─────────────────────────────────────────────────────────
    output_root = os.path.join(output_root, data_flag, time.strftime("%y%m%d_%H%M%S"))
    os.makedirs(output_root, exist_ok=True)

    # ── Data transforms ───────────────────────────────────────────────────────
    print('==> Preparing data...')
    if resize:
        data_transform_train = transforms.Compose([
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=20),
            transforms.RandomAffine(degrees=0, translate=(0.08, 0.08), shear=5),
            transforms.ColorJitter(brightness=0.3, contrast=0.3),
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.3),
            transforms.ToTensor(),
            transforms.Normalize(mean=[.5], std=[.5]),
        ])
        data_transform_val_test = transforms.Compose([
            transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize(mean=[.5], std=[.5]),
        ])
    else:
        data_transform_train = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[.5], std=[.5]),
        ])
        data_transform_val_test = data_transform_train

    # ── Datasets & loaders ────────────────────────────────────────────────────
    train_dataset = DataClass(split='train', transform=data_transform_train,
                              download=download, as_rgb=as_rgb, size=size)
    val_dataset   = DataClass(split='val',   transform=data_transform_val_test,
                              download=download, as_rgb=as_rgb, size=size)
    test_dataset  = DataClass(split='test',  transform=data_transform_val_test,
                              download=download, as_rgb=as_rgb, size=size)

    train_loader         = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    train_loader_at_eval = data.DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
    val_loader           = data.DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
    test_loader          = data.DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

    # ── Build model ───────────────────────────────────────────────────────────
    print('==> Building model...')
    if model_flag == 'medmamba_t':
        model = medmamba_t(num_classes=n_classes)
    elif model_flag == 'medvit_small':
        model = MedViT_small(num_classes=n_classes)
    else:
        raise NotImplementedError(f"model_flag '{model_flag}' chưa được hỗ trợ.")
    model = model.to(device)

    # ── Hardware profile (chạy 1 lần) ─────────────────────────────────────────
    profile_input_size = (
        3 if as_rgb else info['n_channels'],
        224 if resize else size,
        224 if resize else size,
    )
    hw_profile = profile_model(model, profile_input_size, device)
    save_profile_to_txt(hw_profile, output_root, data_flag)

    # Ghi tóm tắt profile lên TensorBoard tab "Text"
    _tb_tmp = SummaryWriter(log_dir=os.path.join(output_root, 'Tensorboard_Results'))
    _tb_tmp.add_text(
        'model_profile',
        f"Params: {hw_profile.get('total_M', 0):.3f} M  |  "
        f"MACs: {hw_profile.get('macs_str', 'N/A')}  |  "
        f"FLOPs: {hw_profile.get('flops_str', 'N/A')}  |  "
        f"Inference: {hw_profile.get('infer_mean_ms', 0):.3f} ms/img",
        global_step=0,
    )
    _tb_tmp.close()

    # ── Evaluators, loss, optimizer, scheduler ────────────────────────────────
    train_evaluator = medmnist.Evaluator(data_flag, 'train', size=size)
    val_evaluator   = medmnist.Evaluator(data_flag, 'val',   size=size)
    test_evaluator  = medmnist.Evaluator(data_flag, 'test',  size=size)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=milestones, gamma=gamma)

    # ── State ─────────────────────────────────────────────────────────────────
    best_auc    = 0
    best_epoch  = 0
    start_epoch = 0
    best_model  = deepcopy(model)

    # ── Resume checkpoint ─────────────────────────────────────────────────────
    if resume_path is not None and os.path.exists(resume_path):
        print(f"🔄 Resuming from: {resume_path}")
        ckpt = torch.load(resume_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_auc    = ckpt['best_auc']
        best_epoch  = ckpt['best_epoch']
        print(f"   -> Epoch {start_epoch}, best AUC: {best_auc:.5f}")

    elif model_path is not None:
        model.load_state_dict(torch.load(model_path, map_location=device)['net'], strict=True)
        for split, evaluator, loader in [
            ('train', train_evaluator, train_loader_at_eval),
            ('val',   val_evaluator,   val_loader),
            ('test',  test_evaluator,  test_loader),
        ]:
            metrics, _, _ = test(model, evaluator, loader, task, criterion, device, run, output_root)
            print(f'{split}  auc: {metrics[1]:.5f}  acc: {metrics[2]:.5f}')

    if num_epochs == 0:
        return

    # ── Log setup ─────────────────────────────────────────────────────────────
    logs       = ['loss', 'auc', 'acc']
    train_logs = ['train_' + l for l in logs]
    val_logs   = ['val_'   + l for l in logs]
    test_logs  = ['test_'  + l for l in logs]
    log_dict   = OrderedDict.fromkeys(train_logs + val_logs + test_logs, 0)

    writer        = SummaryWriter(log_dir=os.path.join(output_root, 'Tensorboard_Results'))
    patience      = 25
    trigger_times = 0

    global iteration
    iteration = 0

    # ── Training loop ─────────────────────────────────────────────────────────
    for epoch in trange(start_epoch, num_epochs):

        # --- Train ---
        train_loss = train(model, train_loader, task, criterion, optimizer, device, writer)

        # --- Evaluate ---
        train_metrics, tr_true, tr_sc = test(model, train_evaluator, train_loader_at_eval,
                                              task, criterion, device, run)
        val_metrics,   vl_true, vl_sc = test(model, val_evaluator,   val_loader,
                                              task, criterion, device, run)
        test_metrics,  ts_true, ts_sc = test(model, test_evaluator,  test_loader,
                                              task, criterion, device, run)
        scheduler.step()

        # --- Extra classification metrics ---
        extra_train = compute_extra_metrics(tr_true, tr_sc, task, n_classes)
        extra_val   = compute_extra_metrics(vl_true, vl_sc, task, n_classes)
        extra_test  = compute_extra_metrics(ts_true, ts_sc, task, n_classes)

        # --- Print console ---
        print(f'\n── Epoch {epoch} ' + '─' * 38)
        print(f'  Train | AUC: {train_metrics[1]:.5f}  ACC: {train_metrics[2]:.5f}'
              f'  F1: {extra_train["f1_macro"]:.5f}  Sens: {extra_train["sensitivity"]:.5f}')
        print(f'  Val   | AUC: {val_metrics[1]:.5f}  ACC: {val_metrics[2]:.5f}'
              f'  F1: {extra_val["f1_macro"]:.5f}  Sens: {extra_val["sensitivity"]:.5f}')
        if extra_val.get('specificity_macro') is not None:
            print(f'         Spec: {extra_val["specificity_macro"]:.5f}'
                  f'  Kappa: {extra_val.get("cohen_kappa", 0):.5f}'
                  f'  MCC: {extra_val.get("mcc", 0):.5f}')

        # --- TensorBoard: AUC / ACC / Loss ---
        for i, key in enumerate(train_logs): log_dict[key] = train_metrics[i]
        for i, key in enumerate(val_logs):   log_dict[key] = val_metrics[i]
        for i, key in enumerate(test_logs):  log_dict[key] = test_metrics[i]
        for key, value in log_dict.items():
            writer.add_scalar(key, value, epoch)

        # --- TensorBoard: extra metrics ---
        for tag, em in [('train', extra_train), ('val', extra_val), ('test', extra_test)]:
            writer.add_scalar(f'{tag}_f1_macro',    em['f1_macro'],        epoch)
            writer.add_scalar(f'{tag}_f1_weighted', em['f1_weighted'],     epoch)
            writer.add_scalar(f'{tag}_precision',   em['precision_macro'], epoch)
            writer.add_scalar(f'{tag}_recall',      em['recall_macro'],    epoch)
            if em.get('specificity_macro') is not None:
                writer.add_scalar(f'{tag}_specificity', em['specificity_macro'], epoch)
            if em.get('cohen_kappa') is not None:
                writer.add_scalar(f'{tag}_cohen_kappa', em['cohen_kappa'], epoch)
            if em.get('mcc') is not None:
                writer.add_scalar(f'{tag}_mcc', em['mcc'], epoch)

        # --- Ghi file log ---
        save_metrics_to_txt(log_dict, extra_train, extra_val, extra_test,
                            epoch, data_flag, output_root)

        # --- Best model & early stopping ---
        cur_auc = val_metrics[1]
        if cur_auc > best_auc:
            best_epoch = epoch
            best_auc   = cur_auc
            best_model = deepcopy(model)
            print(f'  ★ New best AUC: {best_auc:.5f}  (epoch {best_epoch})')

            ckpt_path = os.path.join(output_root, 'best_model_checkpoint.pth')
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_auc':             best_auc,
                'best_epoch':           best_epoch,
                'log_dict':             log_dict,
                'best_extra_val':       extra_val,
                'best_extra_test':      extra_test,
                'hw_profile':           hw_profile,
            }, ckpt_path)
            display(FileLink(os.path.relpath(ckpt_path, os.getcwd())))
            trigger_times = 0
        else:
            trigger_times += 1
            print(f'  EarlyStopping: {trigger_times}/{patience}')
            if trigger_times >= patience:
                print(f'  Early stopping tại epoch {epoch + 1}.')
                break

        # --- Safety checkpoint mỗi 5 epoch ---
        if (epoch + 1) % 5 == 0:
            latest_path = os.path.join(output_root, 'latest_checkpoint.pth')
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_auc':             best_auc,
                'best_epoch':           best_epoch,
            }, latest_path)
            print(f"  💾 Safety checkpoint: epoch {epoch + 1}")

    # ── Final evaluation ───────────────────────────────────────────────────────
    torch.save({'net': best_model.state_dict()},
               os.path.join(output_root, 'best_model.pth'))

    print('\n' + '═' * 55)
    print(f'FINAL RESULTS  —  best model  (epoch {best_epoch})')
    print('═' * 55)
    print(f"  Params    : {hw_profile.get('total_M', 0):.3f} M"
          f"  (trainable: {hw_profile.get('trainable_M', 0):.3f} M)")
    if 'macs_str' in hw_profile:
        print(f"  MACs      : {hw_profile['macs_str']}   FLOPs: {hw_profile['flops_str']}")
    if 'infer_mean_ms' in hw_profile:
        print(f"  Inference : {hw_profile['infer_mean_ms']:.3f}"
              f" ± {hw_profile['infer_std_ms']:.3f} ms/image")
    print('─' * 55)

    final_lines = [f'{data_flag}  (best epoch: {best_epoch})\n']

    for split, evaluator, loader in [
        ('train', train_evaluator, train_loader_at_eval),
        ('val',   val_evaluator,   val_loader),
        ('test',  test_evaluator,  test_loader),
    ]:
        metrics, y_true, y_sc = test(best_model, evaluator, loader,
                                     task, criterion, device, run, output_root)
        extra = compute_extra_metrics(y_true, y_sc, task, n_classes)

        line = (f'{split:<5} | AUC: {metrics[1]:.5f}  ACC: {metrics[2]:.5f}'
                f'  F1(M): {extra["f1_macro"]:.5f}  F1(W): {extra["f1_weighted"]:.5f}'
                f'  Prec: {extra["precision_macro"]:.5f}  Rec: {extra["recall_macro"]:.5f}')
        if extra.get('specificity_macro') is not None:
            line += f'  Spec: {extra["specificity_macro"]:.5f}'
        if extra.get('cohen_kappa') is not None:
            line += f'  Kappa: {extra["cohen_kappa"]:.5f}'
        if extra.get('mcc') is not None:
            line += f'  MCC: {extra["mcc"]:.5f}'

        print(line)
        final_lines.append(line + '\n')

        # Confusion matrix (test set)
        if split == 'test' and extra.get('confusion_matrix') is not None:
            cm = np.array(extra['confusion_matrix'])
            print(f'\nConfusion Matrix (test):\n{cm}')
            final_lines.append(f'\nConfusion Matrix:\n{cm}\n')

        # Classification report (test set)
        if split == 'test' and task != 'multi-label, binary-class':
            y_pred      = np.argmax(y_sc, axis=1)
            label_names = list(info['label'].values())
            report      = classification_report(y_true, y_pred,
                                                target_names=label_names,
                                                zero_division=0)
            print(f'\nClassification Report (test):\n{report}')
            final_lines.append(f'\nClassification Report:\n{report}\n')

    with open(os.path.join(output_root, f'{data_flag}_log.txt'), 'a') as f:
        f.write('\n=== FINAL RESULTS ===\n')
        f.writelines(final_lines)

    writer.close()
    print(f'\n✅ Logs đã lưu tại: {output_root}')


print("✅ Cell 2: Hàm main() đã sẵn sàng.")

In [ ]:
data_flag = 'breastmnist'
output_root = '/kaggle/working/output' 
num_epochs = 100
size = 28
gpu_ids = '0'
batch_size = 32
download = True
#model_flag
resize = True
as_rgb = True
model_path = None
run = 'model1'
model_flags = ['medmamba_t']

for model_flag in model_flags:
    print(f'\n{"="*55}')
    print(f'  Bắt đầu training: {model_flag}  |  dataset: {data_flag}')
    print(f'{"="*55}')
 
    main(
        data_flag   = data_flag,
        output_root = output_root,
        num_epochs  = num_epochs,
        gpu_ids     = gpu_ids,
        batch_size  = batch_size,
        size        = size,
        download    = download,
        model_flag  = model_flag,
        resize      = resize,
        as_rgb      = as_rgb,
        model_path  = model_path,
        run         = f'{model_flag}_run'
    )
 
    print(f'\n✅ Hoàn tất: {model_flag}\n')